In [2]:
from os import listdir, makedirs
from os.path import isfile, isdir, join

import numpy as np
import pandas as pd



noaa_ar = pd.read_csv('noaa_ars.csv')
noaa_ar = noaa_ar.rename(columns = {'Unnamed: 0':'id'})
noaa_ar = noaa_ar.set_index("id")
noaa_ar['year'] = noaa_ar['year'].astype(str)
noaa_ar['month'] = noaa_ar['month'].astype(str)
noaa_ar['day'] = noaa_ar['day'].astype(str)
noaa_ar['ar_time'] = pd.to_datetime(noaa_ar[['year', 'month', 'day']].apply(lambda x: '-'.join(x), axis=1) )

# noaa_ar.head()


In [7]:
from os import listdir, makedirs
from os.path import isfile, isdir, join

import numpy as np
import pandas as pd

def get_aia_flare_dataframe(file_path):
    """Reads the flare dataframe given in flare file path, downloaded using our script"""
    df = pd.read_csv(file_path, delimiter='\t', parse_dates=True)
    df['end_time'] = fix_timestamps(df['end_time'], format='%Y-%m-%dT%H:%M:%S')
    df['start_time'] = fix_timestamps(df['start_time'], format='%Y-%m-%dT%H:%M:%S')
    df['peak_time'] = fix_timestamps(df['peak_time'], format='%Y-%m-%dT%H:%M:%S')
    return df

def get_goes_flare_dataframe(file_path):
    """Reads the flare dataframe given in flare file path, downloaded using our script"""
    df = pd.read_csv(file_path, delimiter='\t', parse_dates=True)
    df['end_time'] = fix_timestamps(df['end_time'], format='%Y-%m-%d %H:%M:%S')
    df['start_time'] = fix_timestamps(df['start_time'], format='%Y-%m-%d %H:%M:%S')
    df['peak_time'] = fix_timestamps(df['peak_time'], format='%Y-%m-%d %H:%M:%S')
    return df


def fix_timestamps(ds, format=None):
    if format is None:
        ds_fixed = pd.to_datetime(ds, format="%Y%m%d_%H%M%S")
    else:
        ds_fixed = pd.to_datetime(ds, format=format)
    return ds_fixed

def remove_duplicates_from_goes_flares(goes_df):
    # remove all totally duplicated rows from goes df
    duplicates = goes_df.duplicated(subset=['start_time', 'end_time', 'peak_time', 'noaa_active_region', 'goes_class', 'goes_location'], keep='first')
    duplicated = goes_df[duplicates]
    goes_df = goes_df[~duplicates]
    return goes_df, duplicated


def remove_duplicates_from_aia_flares(aia_df):
    duplicates = aia_df.duplicated(subset=['start_time', 'end_time', 'peak_time','noaa_active_region', 'goes_class', 'goes_location'], keep='first')
    duplicated = aia_df[duplicates]
    aia_df = aia_df[~duplicates]
    return aia_df, duplicated

def get_flare_dataframe(file_path):
    """Reads the flare dataframe given in flare file path, downloaded using our script"""
    df = pd.read_csv(file_path, delimiter=',',
                     parse_dates=['start_time', 'end_time', 'peak_time', 'peak_time_aia', 'peak_time_goes'])
    new_columns = df.columns.values
    new_columns[0] = 'flare_id'
    df.columns = new_columns
    df = df.set_index('flare_id')

    # return fix_noaa_ar_numbers( pd.read_csv(file_path, delimiter='\t', parse_dates=True) )
    df = fix_end_times(fix_peak_times(fix_flare_locations(fix_noaa_ar_numbers(df))))
    # df = df[ df['goes_class'] > 'C9.9' ]
    return df

def fix_peak_times(df):
    """ Gets the flare dataframe and consolidates the peaktimes from aia and goes using the following strategy
        if the is a valid peak time from goes, it uses that. If not it uses the aia one"""

    # for each null (nan) peak time, get the peak time from goes
    df.loc[pd.isnull(df['peak_time']), 'peak_time'] = df.loc[pd.isnull(df['peak_time']), 'peak_time_goes']
    
    # for each null (nan) peak time, get the peak time from aia
    df.loc[pd.isnull(df['peak_time']), 'peak_time'] = df.loc[pd.isnull(df['peak_time']), 'peak_time_aia']

    return df

def fix_end_times(df):
    """ Gets the flare dataframe and consolidates the end times from aia and goes using the following strategy
        if the is a valid peak time from goes, it uses that. If not it uses the aia one"""

    # for each null (nan) peak time, get the peak time from goes
    df.loc[pd.isnull(df['end_time']), 'end_time'] = df.loc[pd.isnull(df['end_time']), 'end_time_y']
    
    # for each null (nan) peak time, get the peak time from aia
    df.loc[pd.isnull(df['end_time']), 'end_time'] = df.loc[pd.isnull(df['end_time']), 'end_time_x']


    return df

def fix_noaa_ar_numbers(df):
    """ Gets the flare dataframe and consolidates the noaa ar numbers from aia and goes using the following strategy
        if the is a valid goes active region, that is not 0, it uses that. If not it uses a valid aia number"""

    # set goes noaa numbers as primary source
    df['noaa_active_region'] = df['noaa_active_region_goes']
    # if noaa ar no from goes is zero, set it to NaN
    df.loc[df['noaa_active_region'] == 0, 'noaa_active_region'] = np.nan

    # for each null (nan) noaa ar number, get the ar number from aia
    df.loc[pd.isnull(df['noaa_active_region']), 'noaa_active_region'] = \
        df.loc[pd.isnull(df['noaa_active_region']), 'noaa_active_region_aia']
    # for each noaa ar number that is still zero, set it to nan
    df.loc[df['noaa_active_region'] == 0, 'noaa_active_region'] = np.nan

    # print len(df['noaa_active_region'].unique())
    return df

def fix_flare_locations(df):
    """ Gets the flare dataframe and consolidates the flare locations from aia and goes using the following strategy
        if there is a valid aia location, that is not NaN, it uses that. If not it uses the goes location if valid"""

    # set goes location as primary source
    df['fl_location'] = df['goes_location']

    # for each goes location that is still invalid --(0, 0), set it to nan
    df.loc[df['fl_location'] == '(0, 0)', 'fl_location'] = np.nan

    # create two new columns fl_lat and fl_lon for latitude and longitude
    df = df.reindex(columns=np.append(df.columns.values, ['fl_lat', 'fl_lon']))
    # extract the locations from either aia or goes location strings
    df = df.apply(reformat_locations, axis=1)

    # print df.to_csv('flares_flares.csv')
    return df


def reformat_locations(row):
    gloc = row['fl_location']
    # print pd.isnull(gloc),gloc
    if pd.isnull(gloc):
        row['fl_lon'] = np.nan
        row['fl_lat'] = np.nan
    elif gloc.startswith('POINT'):
        row['fl_lon'] = float(gloc.strip('POINT(').strip(')').split(' ')[0])
        row['fl_lat'] = float(gloc.strip('POINT(').strip(')').split(' ')[1])
    elif gloc.startswith('('):
        row['fl_lon'] = float(gloc.strip('(').strip(')').split(', ')[0])
        row['fl_lat'] = float(gloc.strip('(').strip(')').split(', ')[1])
    else:
        row['fl_lon'] = np.nan
        row['fl_lat'] = np.nan
    return row


def transform_fl_lat_lon(fdf):
    lat = fdf['fl_lat'] 
    lon = fdf['fl_lon']
    fdf['y'] = np.sin(np.deg2rad(lat))
    fdf['x'] = np.sin(np.deg2rad(lon)) * np.cos(np.deg2rad(lat))
    return fdf

flare_file_path = '/home/baydin2/workspace/flarepredictiondata/flare_reading/output/joined_flares.csv'
fdf = get_flare_dataframe(flare_file_path)
aia_flare_path = '/home/baydin2/workspace/flarepredictiondata/flare_reading/datain/aia_flares_all.csv'
aia_fl = get_aia_flare_dataframe(aia_flare_path)

goes_flare_path = '/home/baydin2/workspace/flarepredictiondata/flare_reading/datain/goes_flares_all.csv'
goes_fl = get_goes_flare_dataframe(goes_flare_path)
goes_fl = fix_flare_locations(goes_fl)

goes_fl = goes_fl.apply(reformat_locations, axis=1)
goes_fl = transform_fl_lat_lon(goes_fl)


In [100]:
goes_fl.head(2)

,end_time,event_date,goes_class,goes_location,noaa_active_region,peak_time,start_time,fl_location,fl_lat,fl_lon,y,x,centroids
0,2010-05-01 01:43:00,2010-05-01,C5.7,"(-73, 23)",11067,2010-05-01 01:39:00,2010-05-01 01:34:00,"(-73, 23)",23.0,-73.0,0.390731,-0.880283,"(-81.6429165159173, 23.0)"
1,2010-05-01 05:31:00,2010-05-01,B1.6,"(0, 0)",11064,2010-05-01 05:27:00,2010-05-01 05:23:00,NaN,NaN,NaN,NaN,NaN,"(-24.824108682811833, 15.0)"


In [69]:

def calculate_lon_delta(lat_degree, day):
    alpha = 14.11
    beta = -1.7
    gamma = -2.35

    velocity_in_deg = alpha + beta*(np.sin(np.deg2rad(lat_degree)))**2 + gamma*(np.sin(np.deg2rad(lat_degree)))**4
    delta_lon = velocity_in_deg * day
    return delta_lon
# print calculate_lon_delta(0, -25.5)

def search_noaa(noaa_ar, noaa_no, peak_time):
#     print noaa_no
    if noaa_no == 0:
        return {}
    elif noaa_no < 10000:
        noaa_no += 10000
        
    my_ar = noaa_ar[(noaa_ar['noaa_ar_no']==noaa_no)]
    if my_ar.shape[0] == 0:
        print 'There are no AR from NOAA for \#',noaa_no
        return {}
    else:
        my_ar['diff'] = my_ar["ar_time"] - peak_time
        closest = my_ar[np.abs(my_ar['diff']) == np.abs(my_ar['diff']).min() ]
        closest = closest[np.abs(closest['diff'].values) < np.timedelta64(72, 'h')]
        if closest.shape[0] == 1:
            return closest.to_dict(orient='records')
        else:
            return {}
        
def calc_dist(noaa_ar, ar_no, t, fl_x, fl_y):
    ar_record = search_noaa(noaa_ar, ar_no, t)
    if(len(ar_record)) == 1:
        ar_x = ar_record[0]['x']
        ar_y = ar_record[0]['y']
        dist = np.sqrt((fl_x-ar_x)**2 + (fl_y-ar_y)**2)
    else:
        dist = np.nan
    return dist


def calc_cent_dist(noaa_ar, ar_no, t):
    ar_record = search_noaa(noaa_ar, ar_no, t)
    if(len(ar_record)) == 1:
        ar_x = ar_record[0]['x']
        ar_y = ar_record[0]['y']
        return np.sqrt(ar_x**2 + ar_y**2)
    else:
        return np.nan

    return dist


def distance_to_ar(fdf, noaa_ar):
    return fdf.apply(lambda row: calc_dist(noaa_ar, row['noaa_active_region'],\
                                           row['peak_time'], row['x'], row['y']), axis=1)


def distance_to_cent(fdf, noaa_ar):
    return fdf.apply(lambda row: calc_cent_dist(noaa_ar, row['noaa_active_region'], row['peak_time']), axis=1)



def find_noaa_centroid(noaa_ar, ar_no, t, fl_x, fl_y):
    ''' The ar longitudes can be greater than 90 or less than -90 degrees. 
    '''
    ar_record = search_noaa(noaa_ar, ar_no, t)
#     print ar_record
    if(len(ar_record)) == 1:
        ar_lat = ar_record[0]['latitude']
        ar_lon = ar_record[0]['central_meridian_dist']
        
        t_diff_day = ar_record[0]['diff'] / pd.Timedelta('1 day')
#         print t_diff_day
        return '('+str(ar_lon + calculate_lon_delta(ar_lat, -t_diff_day)) + ', ' + str(ar_lat) + ')'
    else:
        return np.nan

def get_ar_centroids(fdf, noaa_ar):
    return fdf.apply(lambda row: find_noaa_centroid(noaa_ar, row['noaa_active_region'],\
                                           row['peak_time'], row['x'], row['y']), axis=1)

# search_noaa(noaa_ar, 11064, np.datetime64('2010-05-01 01:39:00'))


In [71]:
centroids = get_ar_centroids(goes_fl, noaa_ar)
centroids

/home/baydin2/anaconda2/lib/python2.7/site-packages/ipykernel/__main__.py:24: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: http://pandas.pydata.org/pandas-docs/stable/indexing.html#indexing-view-versus-copy


There are no AR from NOAA for \# 11160
There are no AR from NOAA for \# 11160
There are no AR from NOAA for \# 11160
There are no AR from NOAA for \# 11160
There are no AR from NOAA for \# 11160
There are no AR from NOAA for \# 11171
There are no AR from NOAA for \# 11171
There are no AR from NOAA for \# 11171
There are no AR from NOAA for \# 12705
There are no AR from NOAA for \# 12705


0          (-81.6429165159173, 23.0)
1        (-24.824108682811833, 15.0)
2         (-76.91981049565581, 23.0)
3         (-65.26054066470603, 23.0)
4         (-62.96125984348338, 23.0)
5                                NaN
6                                NaN
7                                NaN
8                                NaN
9                                NaN
10         (45.58812899290819, 18.0)
11                               NaN
12                               NaN
13        (-6.20471749132272, -27.0)
14       (-73.23862884396023, -19.0)
15         (20.18552343137194, 42.0)
16         (20.64161044580768, 42.0)
17        (20.954611338067494, 42.0)
18           (21.813128071123, 42.0)
19        (22.966759931166333, 42.0)
20         (27.20568630062788, 42.0)
21        (27.581287371339663, 42.0)
22        (30.899096829293747, 42.0)
23        (31.641356088081316, 42.0)
24         (33.36733243682832, 42.0)
25        (34.103768974914814, 41.0)
26       (17.605696931598963, -26.0)
2

In [72]:
goes_fl['centroids'] = centroids
goes_fl.head()

,end_time,event_date,goes_class,goes_location,noaa_active_region,peak_time,start_time,fl_location,fl_lat,fl_lon,y,x,centroids
0,2010-05-01 01:43:00,2010-05-01,C5.7,"(-73, 23)",11067,2010-05-01 01:39:00,2010-05-01 01:34:00,"(-73, 23)",23.0,-73.0,0.390731,-0.880283,"(-81.6429165159173, 23.0)"
1,2010-05-01 05:31:00,2010-05-01,B1.6,"(0, 0)",11064,2010-05-01 05:27:00,2010-05-01 05:23:00,NaN,NaN,NaN,NaN,NaN,"(-24.824108682811833, 15.0)"
2,2010-05-01 09:59:00,2010-05-01,B1.0,"(0, 0)",11067,2010-05-01 09:52:00,2010-05-01 09:48:00,NaN,NaN,NaN,NaN,NaN,"(-76.91981049565581, 23.0)"
3,2010-05-02 06:18:00,2010-05-02,B2.9,"(0, 0)",11067,2010-05-02 06:09:00,2010-05-02 06:03:00,NaN,NaN,NaN,NaN,NaN,"(-65.26054066470603, 23.0)"
4,2010-05-02 10:15:00,2010-05-02,B3.8,"(0, 0)",11067,2010-05-02 10:09:00,2010-05-02 10:02:00,NaN,NaN,NaN,NaN,NaN,"(-62.96125984348338, 23.0)"


In [73]:
goes_fl.to_csv('./goes_fl_with_centroids.csv')

In [78]:
no_location_fl = goes_fl[(goes_fl['goes_location'] == '(0, 0)') & (pd.isna(goes_fl['centroids']))]

In [83]:
no_location_fl.to_csv('./goes_fl_no_loc.csv')

In [94]:
goes_fl2 = goes_fl[(goes_fl['goes_location'] != '(0, 0)') | (~pd.isna(goes_fl['centroids']))]


In [95]:
def fix_cent_locations(df):
    # set goes location as primary source
    df['fl_location'] = df['goes_location']

    # for each goes location that is still invalid --(0, 0), set it to nan
    df.loc[df['fl_location'] == '(0, 0)', 'fl_location'] = df.loc[df['fl_location'] == '(0, 0)', 'centroids']

    # create two new columns fl_lat and fl_lon for latitude and longitude
    df = df.reindex(columns=np.append(df.columns.values, ['fl_lat', 'fl_lon']))
    # extract the locations from either aia or goes location strings
    df = df.apply(reformat_locations, axis=1)

    # print df.to_csv('flares_flares.csv')
    return df

fix_cent_locations(goes_fl2)
goes_fl2 = goes_fl2.apply(reformat_locations, axis=1)
goes_fl2 = transform_fl_lat_lon(goes_fl2)


/home/baydin2/anaconda2/lib/python2.7/site-packages/ipykernel/__main__.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: http://pandas.pydata.org/pandas-docs/stable/indexing.html#indexing-view-versus-copy
  app.launch_new_instance()


In [98]:
goes_fl2.to_csv('goes_fl_wloc.csv')